# Give an agent MCP tools

This notebook deploys `web-researcher`, an agent that fetches web pages through the Fetch MCP server. See the [README](./README.md) for how kagent connects to workspace MCP servers and the same steps in a terminal.

kagent and MCP servers must be enabled on your prokube cluster.

In [ ]:
%run -n ../../src/pk_helpers/api_key.py
%run -n ../../src/pk_helpers/kagent.py

import subprocess
from pathlib import Path

with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
    namespace = f.read().strip()


def kubectl_apply(manifest: str) -> None:
    subprocess.run(["kubectl", "apply", "-f", "-"], input=manifest, text=True, check=True)

## Prerequisites

The agent reuses the ModelConfig from [`basic-agent`](../basic-agent/) and the Fetch server from [`mcp-servers/deploy-upstream-mcp-server`](../../mcp-servers/deploy-upstream-mcp-server/). The next cells create whatever is missing; `kubectl apply` leaves existing resources unchanged.

The agents call Claude with the key in the `anthropic-api-key` Secret. The next cell creates the Secret unless it already exists, for example because you created it on the **Secrets** page of the prokube UI. It prompts for the key unless `ANTHROPIC_API_KEY` is set.

In [ ]:
secret_exists = subprocess.run(
    ["kubectl", "get", "secret", "anthropic-api-key"], capture_output=True
).returncode == 0
if not secret_exists:
    kubectl_apply(
        Path("../basic-agent/anthropic-secret.yaml")
        .read_text()
        .replace("<your-anthropic-api-key>", get_or_create_anthropic_api_key())
    )

In [ ]:
!kubectl apply -f ../basic-agent/anthropic-model.yaml
!kubectl wait --for=condition=Accepted modelconfigs.kagent.dev/anthropic-haiku --timeout=2m
!kubectl apply -f ../../mcp-servers/deploy-upstream-mcp-server/fetch-server.yaml
!kubectl wait --for=condition=Ready mcpservers.toolhive.stacklok.dev/fetch-example --timeout=3m

## Deploy

prokube publishes the workspace's MCP tools to kagent through the platform-managed `gateway-mcp` `RemoteMCPServer`. When the workspace has more than one MCP server, Agent Gateway prefixes each tool with its server name, so the Fetch tool is `fetch` or `fetch-example_fetch`. The next cell waits until `gateway-mcp` discovers the tool and grants the agent access to that tool only.

In [ ]:
fetch_tool = gateway_mcp_tool_name("fetch-example", "fetch")
kubectl_apply(
    Path("web-researcher.yaml").read_text().replace("- fetch\n", f"- {fetch_tool}\n")
)
!kubectl wait --for=condition=Ready agents.kagent.dev/web-researcher --timeout=3m

## Talk to the agent from this Lab

Labs reach agents through the internal Agent Gateway route without an API key. You can also open **Agents** in the prokube UI, select `web-researcher`, and chat there.

In [ ]:
prompt = "Fetch https://prokube.ai and summarize what prokube offers in one sentence. Include the source URL."
print(send_message(workspace_a2a_url(namespace, "web-researcher"), prompt))

## Connect with an API key

Clients outside the cluster use the agent's external route, `https://<your-prokube-domain>/svc/a2a/<workspace>/web-researcher`, with an API key. Create a key with access to `web-researcher` on the **API Keys** page at `https://<your-prokube-domain>/pkui/ai-gateway/keys`.

The next cell prompts for the key unless `A2A_API_KEY` is set and sends the same message through that route from inside the cluster.

In [ ]:
api_key = get_or_create_a2a_api_key()
print(send_message(agent_a2a_url(namespace, "web-researcher"), prompt, api_key=api_key))

## Clean up

This leaves the Secret, ModelConfig, and Fetch server for the next example.

In [ ]:
!kubectl delete -f web-researcher.yaml